In [23]:
import xml.etree.ElementTree as ET
import re
import csv
from datetime import datetime

In [24]:
def clasificar_metodo_pago(texto):
    texto = texto.lower()

    if "banco pichincha" in texto:
        return "EFECTIVO"
    
    if any(palabra in texto for palabra in ["visa", "titanium", "diners"]):
        return "TARJETA_CREDITO"
    
    return "OTRO"

In [25]:
# Cargar XML
tree = ET.parse("data/sms-backup.xml")
root = tree.getroot()

consumos = []

# Patrones regex
patrones = [
    # Banco Pichincha
    r"trx por ([\d,]+), en ([^,]+)",
    
    # Tarjeta (Diners / Visa)
    r"Consumiste \$ ([\d,]+) en (.+?) con",
]

for sms in root.findall("sms"):
    body = sms.get("body")
    fecha = sms.get("readable_date")
    
    if not body:
        continue

    for patron in patrones:
        match = re.search(patron, body)
        if match:
            monto = match.group(1).replace(",", ".")
            comercio = match.group(2).strip()

            consumos.append({
                "fecha": fecha,
                "monto": float(monto),
                "comercio": comercio,
                "metodo_pago": clasificar_metodo_pago(body),
                "texto": body
            })
            break

# Guardar a CSV
with open("output/consumos.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["fecha", "monto", "comercio", "metodo_pago", "texto"])
    writer.writeheader()
    writer.writerows(consumos)

print(f"Total consumos encontrados: {len(consumos)}")

Total consumos encontrados: 59
